In [2]:
%pwd

'd:\\Euron\\Level2 Projects\\Practice Projects\\End_to_End_Medical_Chatbot-main\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\Euron\\Level2 Projects\\Practice Projects\\End_to_End_Medical_Chatbot-main'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [6]:
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)
    
    documents=loader.load()

    return documents

In [7]:
extracted_data=load_pdf_file(data='Data/')

In [8]:
extracted_data

[Document(metadata={'source': 'Data\\medicalbook.pdf', 'page': 0}, page_content=''),
 Document(metadata={'source': 'Data\\medicalbook.pdf', 'page': 1}, page_content='Elsevier I StudentConsult.com \nTransform the way you learn. \nCompatible with PC, Mac®, most mobile devices, and eReaders, Student Consult \nallows you to browse, search, and interact with this title - online and offline. \nRedeem your PIN at studentconsult.com today! \nStart using these innovative features today: \n• Seamless, real-time integration between devices \n• Straightforward navigation and search \n• Notes and highlights sharing with other users \nthrough social media \n• Enhanced images with annotations, labels, and \nhot spots for zooming on specific details * \n• Live streaming video and animations * \n• Self-assessment tools such as questions \nembedded within the text and multiple-format \nquizzes* \n*some features vary by title \nPIN REDEMPTION INSTRUCTIONS \n1. Login or Sign Up at StudentConsult.com \n2. 

In [9]:
#Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 6165


In [11]:
from langchain.embeddings import HuggingFaceEmbeddings

In [12]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [14]:
pip install -U sentence-transformers

  Using cached sentence_transformers-3.3.1-py3-none-any.whl.metadata (10 kB)
Using cached sentence_transformers-3.3.1-py3-none-any.whl (268 kB)
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2
Note: you may need to restart the kernel to use updated packages.


In [15]:
embeddings = download_hugging_face_embeddings()

d:\Euron\Level2 Projects\Practice Projects\End_to_End_Medical_Chatbot-main\medibot\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [16]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [23]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [25]:
print(PINECONE_API_KEY)

pcsk_h9Zd5_26hstsSXySJK3HKdaU1Zd592F7XCMmPUboCe7N3sbUzad1ggBFjhdMLsg1arsaH


In [26]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"


pc.create_index(
    name=index_name,
    dimension=384, 
    metric="cosine", 
    spec=ServerlessSpec(
        cloud="aws", 
        region="us-east-1"
    ) 
) 

In [27]:
from langchain.vectorstores import Pinecone

docsearch = Pinecone.from_documents(
    documents=text_chunks,  # your document chunks
    index_name=index_name,
    embedding=embeddings
)

In [28]:
docsearch = Pinecone.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [29]:
docsearch

In [32]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":2})

In [33]:
retrieved_docs = retriever.invoke("What is eczema?")

In [34]:
retrieved_docs

[Document(metadata={'page': 957.0, 'source': 'Data\\medicalbook.pdf'}, page_content='oily substance that incre ases the viscosi ty of the tears and \ndecreases the rate of evaporation of tears from the surface \nof the eyeball. Bloc kage and inflammation of a tarsal gland \nis a chalazion and is on the inner surface of the eyelid. \nThe tarsal glands are not the only glands asso ciated \nwith the eyelids. Assoc iated with the eyelash follicles are \nsebaceous and sweat glands (see Fig. 8.7 4). Blockage and \ninflammation of either of these is a stye and is on the edge \nof the eyelid.'),
 Document(metadata={'page': 60.0, 'source': 'Data\\medicalbook.pdf'}, page_content='arranged segment ally along each side of the neural tube \n(Fig. 1.35 ). Part of each somite (the dermatomyotome) \ngives rise to skeletal muscle and the dermis of the skin. As \ncells of the dermatomyotome differentiate, they migrate \ninto posterior (dorsal) and anterior (ventral) areas of the \ndeveloping body: \n• C

In [37]:
from langchain_ollama.llms import OllamaLLM

llm = OllamaLLM(model='gemma:2b')

In [38]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [39]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [40]:
response = rag_chain.invoke({"input": "What is anatomy?"})
print(response["answer"])

Sure, here is a summary of the context.

Anatomy is the study of structures that can be seen with the unaided eye or with the aid of a microscope. It includes gross anatomy, which is the study of structures that can be seen without magnification, and microscopic anatomy, which is the study of cells and tissues using a microscope.


In [41]:
response = rag_chain.invoke({"input": "What is stats?"})
print(response["answer"])

The context does not provide any information about what "stats" is, so I cannot answer this question from the context.
